# Avance Fase 3 — Semana 2

## Núcleo algorítmico, eficiencia y programación orientada a objetos

**Proyecto:** Análisis reproducible de fallas en maquinaria industrial con AI4I 2020  
**Asignatura:** Programación para la Ciencia de Datos  
**Grupo:** 9  
**Estudiante:** Alexander Sepúlveda Ormazábal  

## Introducción

Durante las fases anteriores se definió la problemática del mantenimiento predictivo y se construyó un pipeline reproducible para preparar el conjunto de datos AI4I 2020. Este procesamiento permitió seleccionar las variables analíticas, excluir identificadores y variables que producirían fuga de información, codificar el tipo de producto, escalar las variables operacionales y generar conjuntos separados de entrenamiento y prueba.

En la Fase 3 se reorganiza ese procesamiento mediante programación orientada a objetos. El propósito es conservar la lógica y los resultados de la Fase 2, pero distribuir las responsabilidades en clases y módulos que puedan probarse, reutilizarse y modificarse de manera independiente.

Además, se compararán dos implementaciones de una misma operación para evaluar su tiempo de ejecución y consumo de memoria. Finalmente, se comprobará que la arquitectura orientada a objetos produzca los mismos conjuntos procesados que la implementación de la Fase 2.

## Objetivo general

Reorganizar el pipeline de preprocesamiento de AI4I 2020 mediante programación estructurada y orientada a objetos, conservando los resultados de la Fase 2 y mejorando la modularidad, reutilización, trazabilidad y capacidad de prueba del código.

## Objetivos específicos

1. Identificar las responsabilidades que forman el pipeline de la Fase 2.
2. Distribuir esas responsabilidades en funciones y clases claramente delimitadas.
3. Aplicar encapsulamiento, herencia y polimorfismo en los componentes que lo justifiquen.
4. Comparar dos implementaciones equivalentes mediante mediciones de tiempo y memoria.
5. Determinar si la recursividad aporta una solución adecuada al proyecto o si resulta preferible utilizar división funcional.
6. Seleccionar un patrón de diseño solo si permite resolver un problema concreto de la arquitectura.
7. Probar el funcionamiento del código mediante casos normales, límite y de excepción.
8. Comprobar que los resultados de la Fase 3 coincidan con los archivos procesados en la Fase 2. 

## 1. Verificación del entorno reproducible

Antes de reorganizar el pipeline se verifica que el notebook esté utilizando el entorno virtual del proyecto. También se registran las versiones de Python y de las principales librerías utilizadas.

Esta comprobación permite detectar diferencias de entorno antes de ejecutar el procesamiento. La semilla se mantiene en 42 para reproducir la misma separación de entrenamiento y prueba utilizada en la Fase 2.

In [ ]:
import sys
import platform
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import sklearn

SEMILLA = 42
np.random.seed(SEMILLA)

interprete = Path(sys.executable).resolve()

if ".venv" not in interprete.parts:
    raise RuntimeError(
        "El notebook no está utilizando el entorno virtual .venv del proyecto."
    )

print("Fase 3 · Grupo 9 ·", date.today().isoformat())
print("Intérprete:", interprete)
print("Carpeta de trabajo:", Path.cwd().resolve())
print("Sistema:", platform.platform())
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("Semilla:", SEMILLA)


## 2. Localización del proyecto y definición de rutas

Las rutas del proyecto se construyen a partir de la raíz del repositorio. De esta forma, el notebook no depende de una dirección personal ni de una ubicación específica del computador.

La raíz se reconoce mediante la presencia de `README.md` y de las carpetas `F1` y `F2`. Después se comprueba la existencia del archivo original y de los conjuntos procesados en la Fase 2, ya que estos últimos se utilizarán como referencia para validar la nueva implementación.

In [ ]:
def buscar_raiz(inicio):
    """Busca la raíz del repositorio desde una carpeta inicial."""
    actual = Path(inicio).resolve()

    for candidata in [actual, *actual.parents]:
        tiene_readme = (candidata / "README.md").is_file()
        tiene_f1 = (candidata / "F1").is_dir()
        tiene_f2 = (candidata / "F2").is_dir()

        if tiene_readme and tiene_f1 and tiene_f2:
            return candidata

    raise FileNotFoundError(
        "No se encontró la raíz del repositorio."
    )


RAIZ = buscar_raiz(Path.cwd())

ARCHIVO_CRUDO = (
    RAIZ / "F1" / "data" / "raw" / "ai4i2020.csv"
)

ARCHIVO_ENTRENAMIENTO_F2 = (
    RAIZ
    / "F2"
    / "data"
    / "processed"
    / "ai4i2020_entrenamiento_procesado.csv"
)

ARCHIVO_PRUEBA_F2 = (
    RAIZ
    / "F2"
    / "data"
    / "processed"
    / "ai4i2020_prueba_procesado.csv"
)

DIR_PROCESADO_F3 = RAIZ / "F3" / "data" / "processed"
DIR_DOCS_F3 = RAIZ / "F3" / "docs"
DIR_SRC_F3 = RAIZ / "F3" / "src"

# Estas carpetas se crean solamente si todavía no existen.
for carpeta in [DIR_PROCESADO_F3, DIR_DOCS_F3, DIR_SRC_F3]:
    carpeta.mkdir(parents=True, exist_ok=True)

ARCHIVOS_ENTRADA = {
    "Datos originales": ARCHIVO_CRUDO,
    "Entrenamiento F2": ARCHIVO_ENTRENAMIENTO_F2,
    "Prueba F2": ARCHIVO_PRUEBA_F2,
}

for nombre, ruta in ARCHIVOS_ENTRADA.items():
    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encontró {nombre}: {ruta}"
        )

print("Raíz del repositorio:", RAIZ)

for nombre, ruta in ARCHIVOS_ENTRADA.items():
    print(f"{nombre}: {ruta.relative_to(RAIZ)}")

print(
    "Salida de F3:",
    DIR_PROCESADO_F3.relative_to(RAIZ),
)


## 3. Carga de datos y resultados de referencia

La implementación orientada a objetos debe comenzar desde el archivo original de AI4I 2020 y reproducir el procesamiento realizado en la Fase 2.

Los conjuntos procesados de entrenamiento y prueba se cargan como resultados de referencia. Estos archivos funcionarán como un control para comprobar posteriormente que la reorganización del código no cambió las filas, las columnas ni los valores obtenidos anteriormente.

La carga se encapsula inicialmente en una función con una responsabilidad única: verificar y leer un archivo CSV.

In [ ]:
def cargar_csv(ruta, nombre):
    """Comprueba y carga un archivo CSV."""
    ruta = Path(ruta)

    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encontró {nombre}: {ruta}"
        )

    try:
        datos = pd.read_csv(
            ruta,
            encoding="utf-8-sig",
        )
    except pd.errors.EmptyDataError as error:
        raise ValueError(
            f"El archivo {nombre} está vacío."
        ) from error

    if datos.empty:
        raise ValueError(
            f"El archivo {nombre} no contiene registros."
        )

    print(
        f"{nombre}: "
        f"{datos.shape[0]} filas x "
        f"{datos.shape[1]} columnas"
    )

    return datos


df_crudo = cargar_csv(
    ARCHIVO_CRUDO,
    "datos originales",
)

entrenamiento_f2 = cargar_csv(
    ARCHIVO_ENTRENAMIENTO_F2,
    "entrenamiento de F2",
)

prueba_f2 = cargar_csv(
    ARCHIVO_PRUEBA_F2,
    "prueba de F2",
)

### Verificación de dimensiones

Se comprueba que el conjunto original y los archivos procesados tengan las mismas dimensiones registradas en la Fase 2. Esta validación permite detectar archivos incompletos o diferentes antes de construir el nuevo pipeline.

In [ ]:
DIMENSIONES_ESPERADAS = {
    "datos originales": (10000, 14),
    "entrenamiento de F2": (8000, 9),
    "prueba de F2": (2000, 9),
}

DATOS_CARGADOS = {
    "datos originales": df_crudo,
    "entrenamiento de F2": entrenamiento_f2,
    "prueba de F2": prueba_f2,
}

for nombre, datos in DATOS_CARGADOS.items():
    dimensiones_esperadas = DIMENSIONES_ESPERADAS[nombre]

    if datos.shape != dimensiones_esperadas:
        raise ValueError(
            f"{nombre} tiene dimensiones {datos.shape}, "
            f"pero se esperaban {dimensiones_esperadas}."
        )

print("[OK] Las dimensiones coinciden con la Fase 2.")

### Calidad del conjunto original

Antes de reutilizar el preprocesamiento de la Fase 2 se revisa la presencia de valores ausentes y filas duplicadas.

AI4I 2020 es un conjunto sintético y no presenta valores ausentes ni duplicados completos. Por esta razón no se incorpora una imputación artificial ni se eliminan registros. Esta característica se reconoce como una limitación del conjunto, pero se conserva para evitar modificaciones que no responden a problemas reales de los datos.

In [ ]:
total_nulos = int(
    df_crudo.isna().sum().sum()
)

total_duplicados = int(
    df_crudo.duplicated().sum()
)

print("Valores ausentes:", total_nulos)
print("Filas duplicadas completas:", total_duplicados)

assert total_nulos == 0
assert total_duplicados == 0

print(
    "[OK] No se requiere imputar "
    "ni eliminar registros duplicados."
)

## 4. Descomposición del pipeline de la Fase 2

Antes de crear las clases se identifican las responsabilidades presentes en el pipeline anterior. Esta separación permite decidir qué tareas deben mantenerse como funciones y cuáles necesitan objetos que conserven parámetros entre el ajuste y la transformación.

El procesamiento utiliza cinco variables operacionales y la categoría `Type` como predictores. `Machine failure` se conserva como variable objetivo. Los identificadores se excluyen porque no describen las condiciones de funcionamiento, mientras que los modos de falla se excluyen para evitar fuga de información.

In [ ]:
COLUMNA_OBJETIVO = "Machine failure"
COLUMNA_CATEGORICA = "Type"

COLUMNAS_IDENTIFICADORAS = [
    "UDI",
    "Product ID",
]

COLUMNAS_MODOS_FALLA = [
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF",
]

COLUMNAS_OPERACIONALES = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

COLUMNAS_PREDICTORAS = [
    COLUMNA_CATEGORICA,
    *COLUMNAS_OPERACIONALES,
]

COLUMNAS_CONFIGURADAS = (
    COLUMNAS_IDENTIFICADORAS
    + COLUMNAS_PREDICTORAS
    + [COLUMNA_OBJETIVO]
    + COLUMNAS_MODOS_FALLA
)

columnas_observadas = set(df_crudo.columns)
columnas_configuradas = set(COLUMNAS_CONFIGURADAS)

faltantes = columnas_configuradas - columnas_observadas
no_clasificadas = columnas_observadas - columnas_configuradas

if faltantes:
    raise ValueError(
        f"Faltan columnas esperadas: {sorted(faltantes)}"
    )

if no_clasificadas:
    raise ValueError(
        f"Existen columnas sin un rol definido: {sorted(no_clasificadas)}"
    )

print("[OK] Las 14 variables tienen un rol definido.")
print("Predictores:", COLUMNAS_PREDICTORAS)
print("Objetivo:", COLUMNA_OBJETIVO)
print("Identificadores excluidos:", COLUMNAS_IDENTIFICADORAS)
print("Modos de falla excluidos:", COLUMNAS_MODOS_FALLA)


### Responsabilidades identificadas

El pipeline puede dividirse inicialmente en tres responsabilidades:

1. **Selección de variables:** conserva los predictores y excluye identificadores y modos de falla.
2. **Codificación:** aprende las categorías presentes en `Type` y las transforma en columnas binarias.
3. **Escalamiento:** aprende los parámetros de las variables operacionales desde entrenamiento y los aplica posteriormente a prueba.

La variable objetivo se mantiene separada del pipeline de transformación para evitar que sea modificada y para conservarla como última columna de los conjuntos procesados.

## 5. Implementación orientada a objetos

Las transformaciones que necesitan conservar parámetros entre el ajuste y la aplicación se organizarán mediante clases. Se utilizará una clase base para reunir el comportamiento común y clases hijas para implementar la codificación y el escalamiento.

La selección de variables se mantendrá como una función, ya que no necesita aprender ni conservar parámetros.

### 5.1 Clase base `Transformador`

La clase base reúne el comportamiento compartido por las transformaciones. Cada objeto deberá ajustarse con los datos de entrenamiento antes de transformar un conjunto.

El estado del ajuste y los parámetros aprendidos se mantienen como atributos internos. Las clases hijas deberán definir qué información aprenden y cómo la aplican.

In [ ]:
class Transformador:
    """Clase base para las transformaciones del pipeline."""

    def __init__(self, columna):
        self.columna = columna
        self._parametros = {}
        self._ajustado = False

    def ajustar(self, datos):
        """Aprende los parámetros desde el conjunto de entrenamiento."""
        if self.columna not in datos.columns:
            raise KeyError(f"No existe la columna: {self.columna}")

        self._parametros = self.aprender(datos)
        self._ajustado = True
        return self

    def transformar(self, datos):
        """Aplica los parámetros aprendidos a otro conjunto."""
        if not self._ajustado:
            raise RuntimeError(
                "El transformador debe ajustarse antes de transformar."
            )

        if self.columna not in datos.columns:
            raise KeyError(f"No existe la columna: {self.columna}")

        return self.aplicar(datos.copy())

    def aprender(self, datos):
        """Cada clase hija debe definir qué necesita aprender."""
        raise NotImplementedError

    def aplicar(self, datos):
        """Cada clase hija debe definir cómo transforma los datos."""
        raise NotImplementedError

### 5.2 Codificación de `Type`

Esta clase aprende las categorías de `Type` desde entrenamiento y las transforma en columnas binarias. El codificador aprendido se conserva para aplicar exactamente la misma transformación al conjunto de prueba.

In [ ]:
from sklearn.preprocessing import OneHotEncoder


class CodificadorOneHot(Transformador):
    """Convierte una columna categórica en columnas binarias."""

    def aprender(self, datos):
        codificador = OneHotEncoder(
            sparse_output=False,
            handle_unknown="error",
            dtype=np.int8,
        )

        codificador.fit(datos[[self.columna]])

        return {
            "codificador": codificador,
            "columnas": codificador.get_feature_names_out(
                [self.columna]
            ).tolist(),
        }

    def aplicar(self, datos):
        codificador = self._parametros["codificador"]
        columnas = self._parametros["columnas"]

        matriz = codificador.transform(datos[[self.columna]])

        datos_codificados = pd.DataFrame(
            matriz,
            columns=columnas,
            index=datos.index,
        )

        return pd.concat(
            [
                datos.drop(columns=[self.columna]),
                datos_codificados,
            ],
            axis=1,
        )

### 5.3 Escalamiento de las variables operacionales

La segunda clase hija reproduce el escalamiento robusto seleccionado en la Fase 2. Cada objeto aprende la mediana y el rango intercuartílico de una variable operacional utilizando únicamente el conjunto de entrenamiento.

Los parámetros aprendidos se conservan para aplicar la misma transformación al conjunto de prueba.

In [ ]:
from sklearn.preprocessing import RobustScaler


class EscaladorRobusto(Transformador):
    """Escala una variable mediante la mediana y el rango intercuartílico."""

    def aprender(self, datos):
        escalador = RobustScaler()
        escalador.fit(datos[[self.columna]])

        return {
            "escalador": escalador,
            "mediana": float(escalador.center_[0]),
            "iqr": float(escalador.scale_[0]),
        }

    def aplicar(self, datos):
        escalador = self._parametros["escalador"]

        valores_escalados = escalador.transform(
            datos[[self.columna]]
        )

        datos[self.columna] = valores_escalados.ravel()

        return datos

### 5.4 Separación de predictores y objetivo

La selección se mantiene como una función porque no necesita aprender ni recordar parámetros. Su responsabilidad es separar las variables predictoras de la variable objetivo y comprobar que todas las columnas necesarias estén disponibles.

In [ ]:
def separar_predictores_objetivo(datos, predictores, objetivo):
    """Separa las variables predictoras de la variable objetivo."""

    columnas_necesarias = predictores + [objetivo]

    faltantes = [
        columna
        for columna in columnas_necesarias
        if columna not in datos.columns
    ]

    if faltantes:
        raise KeyError(
            f"Faltan columnas necesarias: {faltantes}"
        )

    X = datos[predictores].copy()
    y = datos[objetivo].copy()

    return X, y

### 5.5 División reproducible de los datos

Los predictores y la variable objetivo se separan antes de dividir el conjunto. Se mantiene la proporción de 80 % para entrenamiento y 20 % para prueba, utilizando la misma semilla y la misma estratificación de la Fase 2.

La estratificación ayuda a conservar una proporción semejante de fallas en ambos conjuntos.



In [ ]:
from sklearn.model_selection import train_test_split


X, y = separar_predictores_objetivo(
    df_crudo,
    COLUMNAS_PREDICTORAS,
    COLUMNA_OBJETIVO,
)

(
    X_entrenamiento_crudo,
    X_prueba_crudo,
    y_entrenamiento,
    y_prueba,
) = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEMILLA,
    stratify=y,
)

print("Entrenamiento:", X_entrenamiento_crudo.shape)
print("Prueba:", X_prueba_crudo.shape)
print("Objetivo de entrenamiento:", y_entrenamiento.shape)
print("Objetivo de prueba:", y_prueba.shape)

## 6. Composición del pipeline

La clase `Pipeline` organiza y ejecuta las transformaciones en el orden definido. Durante el ajuste, cada paso aprende sus parámetros desde entrenamiento. Posteriormente, esos mismos pasos transforman otros conjuntos sin volver a ajustarse.

El pipeline trata de la misma forma al codificador y a los escaladores, ya que todos heredan los métodos `ajustar()` y `transformar()` de la clase base. Este comportamiento corresponde al polimorfismo.

In [ ]:
class Pipeline:
    """Organiza y ejecuta los pasos de transformación."""

    def __init__(self, pasos):
        self._pasos = list(pasos)
        self._ajustado = False

    def ajustar(self, datos):
        intermedio = datos.copy()

        for paso in self._pasos:
            paso.ajustar(intermedio)
            intermedio = paso.transformar(intermedio)

        self._ajustado = True
        return self

    def transformar(self, datos):
        if not self._ajustado:
            raise RuntimeError(
                "El pipeline debe ajustarse antes de transformar."
            )

        resultado = datos.copy()

        for paso in self._pasos:
            resultado = paso.transformar(resultado)

        return resultado

### 6.1 Ejecución del pipeline

El pipeline se construye con un codificador para `Type` y un escalador robusto para cada variable operacional. Todos los pasos se ajustan utilizando el conjunto de entrenamiento.

Después del ajuste, el mismo pipeline transforma entrenamiento y prueba. Finalmente, la variable objetivo se vuelve a incorporar como última columna.

In [ ]:
pasos = [
    CodificadorOneHot(COLUMNA_CATEGORICA),
]

for columna in COLUMNAS_OPERACIONALES:
    pasos.append(EscaladorRobusto(columna))

pipeline_f3 = Pipeline(pasos)

pipeline_f3.ajustar(X_entrenamiento_crudo)

X_entrenamiento_f3 = pipeline_f3.transformar(
    X_entrenamiento_crudo
)

X_prueba_f3 = pipeline_f3.transformar(
    X_prueba_crudo
)

entrenamiento_f3 = X_entrenamiento_f3.copy()
entrenamiento_f3[COLUMNA_OBJETIVO] = y_entrenamiento
entrenamiento_f3 = entrenamiento_f3.reset_index(drop=True)

prueba_f3 = X_prueba_f3.copy()
prueba_f3[COLUMNA_OBJETIVO] = y_prueba
prueba_f3 = prueba_f3.reset_index(drop=True)

print("Cantidad de pasos:", len(pasos))
print("Entrenamiento F3:", entrenamiento_f3.shape)
print("Prueba F3:", prueba_f3.shape)
print("Columnas:", entrenamiento_f3.columns.tolist())


## 7. Verificación respecto de la Fase 2

Se comparan los resultados generados por el pipeline orientado a objetos con los archivos procesados y guardados en la Fase 2.

La comprobación permite demostrar que la nueva arquitectura reorganiza el código sin alterar los resultados del preprocesamiento anterior.

In [ ]:
pd.testing.assert_frame_equal(
    entrenamiento_f3,
    entrenamiento_f2,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

pd.testing.assert_frame_equal(
    prueba_f3,
    prueba_f2,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

print("[OK] Entrenamiento coincide con la Fase 2.")
print("[OK] Prueba coincide con la Fase 2.")
print("[OK] La reorganización no modificó los resultados.")


## 8. Diseño estructurado: cohesión y acoplamiento

### 8.1 Cohesión de los componentes

Un diseño tendría cohesión baja si una sola clase se encargara de cargar archivos, seleccionar variables, codificar categorías, escalar mediciones y guardar resultados. Esa clase concentraría tareas diferentes y sería difícil de probar o modificar.

En este proyecto se utiliza alta cohesión, porque cada componente tiene una responsabilidad concreta:

| Componente | Responsabilidad |
|---|---|
| `cargar_csv` | Verificar y cargar un archivo CSV. |
| `separar_predictores_objetivo` | Separar las variables predictoras y la variable objetivo. |
| `CodificadorOneHot` | Codificar la categoría `Type`. |
| `EscaladorRobusto` | Escalar una variable operacional. |
| `Pipeline` | Ejecutar los pasos en el orden definido. |

Esta separación permite revisar y probar cada parte sin ejecutar tareas que no le corresponden.